# Module 07: Difference-in-Differences as an Interaction Model

This module shows how Difference-in-Differences can be estimated
with linear regression. We start with a 2x2 table, connect it to
interaction terms, then move to panel data, fixed effects, visual
diagnostics, event studies, and applied reporting.

Recommended order:

1. `01_why_before_after_fails.ipynb`
2. `02_did_as_interaction_regression.ipynb`
3. `03_continuous_variables_and_moderation.ipynb`
4. `04_panel_did_fixed_effects.ipynb`
5. `05_parallel_trends_and_event_study.ipynb`
6. `06_inference_threats_robustness.ipynb`
7. `07_transfer_did_report_workflow.ipynb`
8. `08_optional_staggered_timing_preview.ipynb`

Big idea: DiD is a regression interaction model, but the causal
interpretation depends on the comparison group providing a
plausible counterfactual trend.


In [1]:
from lite_setup import ensure_packages
await ensure_packages()

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm
import statsmodels.formula.api as smf

from checks import check_close, check_columns, check_same_estimate, check_sign
from did_utils import (
    did_2x2_table,
    manual_did,
    predicted_values_2x2,
    plot_group_trends,
    plot_did_counterfactual,
    make_event_dummies,
    extract_event_study_results,
    plot_event_study,
)


plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

DATA_DIR = Path("data")


Running outside JupyterLite; assuming packages are already installed.


In [2]:
expected = [
    DATA_DIR / "tutoring_2x2_group.csv",
    DATA_DIR / "tutoring_student_panel.csv",
    DATA_DIR / "retail_rollout_panel.csv",
    DATA_DIR / "platform_exposure_panel.csv",
    DATA_DIR / "trend_scenarios_panel.csv",
    DATA_DIR / "did_capstone_case.csv",
    DATA_DIR / "staggered_rollout_preview.csv",
]

for path in expected:
    df = pd.read_csv(path)
    print(f"{path}: {df.shape[0]} rows, {df.shape[1]} columns")


data\tutoring_2x2_group.csv: 4 rows, 5 columns
data\tutoring_student_panel.csv: 672 rows, 9 columns
data\retail_rollout_panel.csv: 1440 rows, 10 columns
data\platform_exposure_panel.csv: 256 rows, 9 columns
data\trend_scenarios_panel.csv: 1296 rows, 8 columns
data\did_capstone_case.csv: 540 rows, 9 columns
data\staggered_rollout_preview.csv: 1152 rows, 8 columns


## How to work

Run each notebook from top to bottom. The synthetic datasets are
small on purpose: they let you see the treatment effect, the
counterfactual comparison, the fixed effects, and the warning
signs without fighting a large data-cleaning problem.

JupyterLite stores work in browser storage. Download your
completed notebook before switching browsers/devices if you want
to keep your work.
